In [0]:
df_raw =  spark.table("workspace.filestore.ventas_crudas")
df_raw.write.format("delta").mode("overwrite").saveAsTable("workspace.filestore.ventas_bronce")
display(spark.table("workspace.filestore.ventas_bronce"))

invoice_id,branch,city,customer_type,gender,product_line,unit_price,quantity,tax_five_percent,sales,date,time,payment,cogs,gross_margin_percentage,gross_income,rating
750-67-8428,Alex,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,2019-01-05,1:08:00 PM,Ewallet,522.83,4.761904762,26.1415,9.1
226-31-3081,Giza,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.82,80.22,2019-03-08,10:29:00 AM,Cash,76.4,4.761904762,3.82,9.6
631-41-3108,Alex,Yangon,Normal,Female,Home and lifestyle,46.33,7,16.2155,340.5255,2019-03-03,1:23:00 PM,Credit card,324.31,4.761904762,16.2155,7.4
123-19-1176,Alex,Yangon,Member,Female,Health and beauty,58.22,8,23.288,489.048,2019-01-27,8:33:00 PM,Ewallet,465.76,4.761904762,23.288,8.4
373-73-7910,Alex,Yangon,Member,Female,Sports and travel,86.31,7,30.2085,634.3785,2019-02-08,10:37:00 AM,Ewallet,604.17,4.761904762,30.2085,5.3
699-14-3026,Giza,Naypyitaw,Member,Female,Electronic accessories,85.39,7,29.8865,627.6165,2019-03-25,6:30:00 PM,Ewallet,597.73,4.761904762,29.8865,4.1
355-53-5943,Alex,Yangon,Member,Female,Electronic accessories,68.84,6,20.652,433.692,2019-02-25,2:36:00 PM,Ewallet,413.04,4.761904762,20.652,5.8
315-22-5665,Giza,Naypyitaw,Member,Female,Home and lifestyle,73.56,10,36.78,772.38,2019-02-24,11:38:00 AM,Ewallet,735.6,4.761904762,36.78,8.0
665-32-9167,Alex,Yangon,Member,Female,Health and beauty,36.26,2,3.626,76.146,2019-01-10,5:15:00 PM,Credit card,72.52,4.761904762,3.626,7.2
692-92-5582,Cairo,Mandalay,Member,Female,Food and beverages,54.84,3,8.226,172.746,2019-02-20,1:27:00 PM,Credit card,164.52,4.761904762,8.226,5.9


In [0]:
from pyspark.sql.functions import col, when, lit
df_silver = spark.table("workspace.filestore.ventas_bronce")

# Eliminar filas con nulos en columnas clave
df_silver = df_silver.dropna(
    subset=["invoice_id", "gross_income", "customer_type", "rating"]
)

df_silver = df_silver.withColumn(
    "es_miembro",
    when(col("customer_type") == "Member", lit(True)).otherwise(lit(False))
)

df_silver = df_silver.select(
    col("invoice_id"),
    col("branch"),
    col("city"),
    col("customer_type"),
    col("es_miembro"),       # columna nueva
    col("gender"),
    col("product_line"),
    col("unit_price"),
    col("quantity"),
    col("payment"),
    col("date"),
    col("rating"),
    col("gross_income")
)

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.filestore.ventas_plata")

print(f"ventas_plata: {df_silver.count()} filas")
df_silver.show(5)

ventas_plata: 1000 filas
+-----------+------+---------+-------------+----------+------+--------------------+----------+--------+-----------+----------+------+------------+
| invoice_id|branch|     city|customer_type|es_miembro|gender|        product_line|unit_price|quantity|    payment|      date|rating|gross_income|
+-----------+------+---------+-------------+----------+------+--------------------+----------+--------+-----------+----------+------+------------+
|750-67-8428|  Alex|   Yangon|       Member|      true|Female|   Health and beauty|     74.69|       7|    Ewallet|2019-01-05|   9.1|     26.1415|
|226-31-3081|  Giza|Naypyitaw|       Normal|     false|Female|Electronic access...|     15.28|       5|       Cash|2019-03-08|   9.6|        3.82|
|631-41-3108|  Alex|   Yangon|       Normal|     false|Female|  Home and lifestyle|     46.33|       7|Credit card|2019-03-03|   7.4|     16.2155|
|123-19-1176|  Alex|   Yangon|       Member|      true|Female|   Health and beauty|     58.22

In [0]:
from pyspark.sql.functions import sum as _sum, avg, count, round

df_plata = spark.table("workspace.filestore.ventas_plata")

df_gold = df_plata.groupBy("product_line").agg(
    round(_sum("gross_income"), 2).alias("ingreso_total"),
    round(avg("rating"), 2).alias("calificacion_promedio"),
    count("invoice_id").alias("volumen_transacciones")
).orderBy("ingreso_total", ascending=False)

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.filestore.ventas_gold")

print("ventas_gold guardada")
df_gold.show()

ventas_gold guardada
+--------------------+-------------+---------------------+---------------------+
|        product_line|ingreso_total|calificacion_promedio|volumen_transacciones|
+--------------------+-------------+---------------------+---------------------+
|  Food and beverages|      2673.56|                 7.11|                  174|
|   Sports and travel|       2624.9|                 6.92|                  166|
|Electronic access...|       2587.5|                 6.92|                  170|
| Fashion accessories|       2586.0|                 7.03|                  178|
|  Home and lifestyle|      2564.85|                 6.84|                  160|
|   Health and beauty|      2342.56|                  7.0|                  152|
+--------------------+-------------+---------------------+---------------------+

